In [2]:
# Actualizar repositorios e instalar Java
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless -qq -y

# Descargar Spark (usando versión estable del archivo)
!wget -q https://archive.apache.org/dist/spark/spark-3.5.7/spark-3.5.7-bin-hadoop3.tgz
!tar xf spark-3.5.7-bin-hadoop3.tgz

# Instalar findspark
!pip install -q findspark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libxtst6:amd64.
(Reading database ... 121713 files and directories currently installed.)
Preparing to unpack .../libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package openjdk-8-jre-headless:amd64.
Preparing to unpack .../openjdk-8-jre-headless_8u472-ga-1~22.04_amd64.deb ...
Unpacking openjdk-8-jre-headless:amd64 (8u472-ga-1~22.04) ...
Selecting previously unselected package openjdk-8-jdk-headless:amd64.
Preparing to unpack .../openjdk-8-jdk-headless_8u472-ga-1~22.04_amd64.deb ...
Unpacking openjdk-8-jdk-headless:amd64 (8u472-ga-1~22.04) ...
Setting up libxtst6:amd64 (2:1.2.3-1build4) ...
Setting up openjdk-8-jre-headless:amd64 (8u472-ga-1~22.04) ...
update-alternatives: using /usr/lib/jvm

In [3]:
!pip install opendatasets
import opendatasets as od

In [4]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [10]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.7-bin-hadoop3"

In [11]:
!ls

sample_data  spark-3.5.7-bin-hadoop3  spark-3.5.7-bin-hadoop3.tgz


In [12]:
import findspark
findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) 
spark

In [ ]:
# from google.colab import drive  # type: ignore
# drive.mount('/content/drive')

In [13]:
# dataset_link="https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store"
dataset_link="https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store?select=2019-Nov.csv"
od.download(dataset_link)

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username:Your Kaggle Key:Dataset URL: https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store


100%|██████████| 4.29G/4.29G [00:59<00:00, 77.1MB/s]

In [14]:
import os
os.chdir("ecommerce-behavior-data-from-multi-category-store")
os.listdir()

['spark-3.5.7-bin-hadoop3.tgz',
 'spark-3.5.7-bin-hadoop3',
 '2019-Nov.csv',
 '2019-Oct.csv']

# 1. Análisis profundo de los datos


## 1.1 Estructura de los datos

In [16]:
# Lectura con esquema explícito (evita inferSchema que escanea 2 veces)
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", LongType(), True),
    StructField("category_id", LongType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("user_id", LongType(), True),
    StructField("user_session", StringType(), True)
])

df = spark.read.csv('ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv', header=True, schema=schema)
df.show(5)
print(f"Columnas: {df.columns}")

+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|2053013555631882655|electr

In [53]:
# Esquema del DataFrame
df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



In [13]:
df.describe()

summary,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
count,42448764,42448764,42448764,28933155,36335756,42448764,42448764,42448762
mean,NULL,1.0549932375842676E7,2.057404237936260...,NULL,NaN,290.3236606848809,5.335371475081686E8,NULL
stddev,NULL,1.1881906970608136E7,1.843926466140411...,NULL,NaN,358.2691553394021,1.852373817465431E7,NULL
min,cart,1000978,2053013552226107603,accessories.bag,a-case,0.0,33869381,00000042-3e3f-42f...
max,view,60500010,2175419595093967522,stationery.cartrige,zyxel,2574.07,566280860,fffffc65-7ce9-435...


__Valores únicos por columna__

In [15]:
from pyspark.sql.functions import countDistinct

df.agg(
    countDistinct("event_type").alias("valores_unicos_event_type"),
    countDistinct("product_id").alias("valores_unicos_product_id"),
    countDistinct("category_id").alias("valores_unicos_category_id"),
    countDistinct("category_code").alias("valores_unicos_category_code"),
    countDistinct("brand").alias("valores_unicos_brand"),
    countDistinct("price").alias("valores_unicos_price"),
    countDistinct("user_id").alias("valores_unicos_user_id"),
).show()

+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+
|valores_unicos_event_type|valores_unicos_product_id|valores_unicos_category_id|valores_unicos_category_code|valores_unicos_brand|valores_unicos_price|valores_unicos_user_id|
+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+
|                        3|                   166794|                       624|                         126|                3445|               65298|               3022290|
+-------------------------+-------------------------+--------------------------+----------------------------+--------------------+--------------------+----------------------+



__Registros duplicados__

In [ ]:
# Duplicados en una sola pasada (reutiliza para limpieza)
df_sin_dup = df.dropDuplicates()
duplicados = df.count() - df_sin_dup.count()
print(f"Duplicados: {duplicados}")

In [18]:
# Reutiliza df_sin_dup de la celda anterior + limpieza de nulos en una sola operación
from pyspark.sql.functions import when, col, coalesce, lit

df_clean = df_sin_dup.withColumn(
    "category_code_clean", coalesce(col("category_code"), lit("unknown"))
).withColumn(
    "brand_clean", coalesce(col("brand"), lit("no_brand"))
)

**Reemplazo de valores nulos en columnas category_code y brand** (Se les reasigna un nuevo valor a los registros con valores nulos)

In [19]:
from pyspark.sql.functions import when, col

df_clean = df_clean.withColumn(
    "category_code_clean",
    when(col("category_code").isNull(), "unknown").otherwise(col("category_code"))
)

In [20]:
# Cache estratégico después de limpieza
df_clean.cache()
print(f"Registros en df_clean: {df_clean.count()}")

In [59]:
df_clean.show(10)

+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+--------------------+-----------+
|         event_time|event_type|product_id|        category_id|       category_code|  brand| price|  user_id|        user_session| category_code_clean|brand_clean|
+-------------------+----------+----------+-------------------+--------------------+-------+------+---------+--------------------+--------------------+-----------+
|2019-10-01 00:06:50|      view|   3700766|2053013565983425517|appliances.enviro...|samsung|128.45|555448298|a4378fe7-5e6a-4b9...|appliances.enviro...|    samsung|
|2019-10-01 00:16:16|      view|  28401058|2053013566209917945|     accessories.bag|  karya|100.39|547093079|6619868d-16c8-401...|     accessories.bag|      karya|
|2019-10-01 02:17:52|      view|   1004659|2053013555631882655|electronics.smart...|samsung|787.18|519447668|264c64cd-e48d-44d...|electronics.smart...|    samsung|
|2019-10-01 02:1

## 1.2 Análisis temporal

In [35]:
from pyspark.sql.functions import min, max

df_clean.select(
    min("event_time").alias("fecha_minima"),
    max("event_time").alias("fecha_maxima")
).show()

+-------------------+-------------------+
|       fecha_minima|       fecha_maxima|
+-------------------+-------------------+
|2019-10-01 00:00:00|2019-10-31 23:59:59|
+-------------------+-------------------+



In [52]:
from pyspark.sql.functions import to_date, hour, dayofweek

# Análisis temporal en una sola pasada con cache compartido
analisis_temporal = df_clean.select(
    dayofweek(col("event_time")).alias("dia_semana"),
    hour(col("event_time")).alias("hora_dia"),
    to_date(col("event_time")).alias("fecha")
).cache()

print("Eventos por día de la semana:")
analisis_temporal.groupBy("dia_semana").count().orderBy("dia_semana").show()

+----------+-------+
|dia_semana|  count|
+----------+-------+
|         1|5851611|
|         2|5317662|
|         3|6797348|
|         4|6648353|
|         5|6376063|
|         6|5824835|
|         7|5602672|
+----------+-------+



In [ ]:
# Reutiliza cache de analisis_temporal
analisis_temporal.groupBy("hora_dia").count().orderBy(col("count").desc()).show(10)

+--------+-------+
|hora_dia|  count|
+--------+-------+
|      16|3053226|
|      15|2979082|
|      17|2732443|
|      14|2676546|
|       8|2387991|
|      13|2353321|
|       9|2349336|
|       7|2333480|
|      10|2295245|
|       6|2266954|
+--------+-------+
only showing top 10 rows



In [ ]:
# Reutiliza cache
eventos_por_dia = analisis_temporal.groupBy("fecha").count()
eventos_por_dia.describe("count").show()

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|                31|
|   mean|1368340.1290322582|
| stddev|120140.19894682542|
|    min|           1126624|
|    max|           1638290|
+-------+------------------+



## 1.3 Análisis de comportamiento de usuarios

**Distribución de tipos de eventos**

In [ ]:
# Distribución de eventos con cache para reutilizar en tasas de conversión
from pyspark.sql.functions import sum as spark_sum

distribucion_eventos = df_clean.groupBy("event_type").count().cache()
total_eventos = distribucion_eventos.agg(spark_sum("count")).collect()[0][0]

distribucion_eventos.withColumn("porcentaje", col("count") / total_eventos * 100) \
    .orderBy(col("count").desc()).show()

+----------+--------+------------------+
|event_type|   count|        porcentaje|
+----------+--------+------------------+
|      view|40777328|  96.1308997310233|
|      cart|  898443|2.1180429955351605|
|  purchase|  742773|1.7510572734415402|
+----------+--------+------------------+



In [56]:
# Reutiliza distribucion_eventos cacheado (evita 4 count() separados)
conteos = {row["event_type"]: row["count"] for row in distribucion_eventos.collect()}
views = conteos.get("view", 0)
carts = conteos.get("cart", 0)
purchases = conteos.get("purchase", 0)

print("Tasas de conversión globales:")
print(f"Views: {views:,} ({views/total_eventos*100:.2f}%)")
print(f"Carts: {carts:,} ({carts/total_eventos*100:.2f}%)")
print(f"Purchase: {purchases:,} ({purchases/total_eventos*100:.2f}%)")
print(f"\nTasa cart/view: {carts/views*100:.2f}%")
print(f"Tasa purchase/view: {purchases/views*100:.2f}%")
print(f"Tasa purchase/cart: {purchases/carts*100:.2f}%")

Tasas de conversión globales: 
Views: 40,777,328 (96.13%)
Carts: 898,443 (2.12%)
Purchase: 742,773 (1.75%)

Tasa cart/view: 2.20%
Tasa purchase/view: 1.82%
Tasa purchase/cart: 82.67%


**Métricas de actividad por usuario**

In [24]:
# Reutiliza eventos_por_tipo_usuario 
# Ejecutar primero eventos por tipo (celda de abajo)
eventos_por_usuario = eventos_por_tipo_usuario.withColumn(
    "total_eventos", col("cart") + col("purchase") + col("view")
)
eventos_por_usuario.describe("total_eventos").show()

+-------+-----------------+
|summary|    total_eventos|
+-------+-----------------+
|  count|          3022290|
|   mean| 14.0352328863213|
| stddev|32.75705325743597|
|    min|                1|
|    max|             7436|
+-------+-----------------+



In [58]:
# Distribución en percentiles
eventos_por_usuario.selectExpr(
    "percentile_approx(total_eventos, 0.25) as p25",
    "percentile_approx(total_eventos, 0.50) as p50_median",
    "percentile_approx(total_eventos, 0.75) as p75",
    "percentile_approx(total_eventos, 0.90) as p90",
    "percentile_approx(total_eventos, 0.95) as p95",
    "percentile_approx(total_eventos, 0.99) as p99"
).show()

+---+----------+---+---+---+---+
|p25|p50_median|p75|p90|p95|p99|
+---+----------+---+---+---+---+
|  2|         4| 13| 34| 57|140|
+---+----------+---+---+---+---+



**Eventos por tipo**

In [29]:
# Pivot directo (evita doble groupBy) + cache para múltiples usos
eventos_por_tipo_usuario = df_clean.groupBy("user_id").pivot("event_type").count().fillna(0).cache()
eventos_por_tipo_usuario.show(5)

+---------+----+--------+----+
|  user_id|cart|purchase|view|
+---------+----+--------+----+
|514700043|   0|       0|   4|
|523367237|   1|       1|  53|
|519705698|   0|       0|  89|
|519798460|   1|       2| 102|
|547283407|   3|       3|  13|
+---------+----+--------+----+
only showing top 5 rows



In [ ]:
# Una sola llamada describe para los 3 tipos
eventos_por_tipo_usuario.select("view", "cart", "purchase").describe().show()

+-------+------------------+
|summary|              view|
+-------+------------------+
|  count|           3022290|
|   mean|13.492195652965135|
| stddev|31.855348368395525|
|    min|                 0|
|    max|              7436|
+-------+------------------+

+-------+------------------+
|summary|              cart|
+-------+------------------+
|  count|           3022290|
|   mean|0.2972722670557756|
| stddev|1.5011636291005397|
|    min|                 0|
|    max|               356|
+-------+------------------+

+-------+------------------+
|summary|          purchase|
+-------+------------------+
|  count|           3022290|
|   mean|0.2457649663003881|
| stddev|1.4093212679874512|
|    min|                 0|
|    max|               321|
+-------+------------------+



**Compradores vs navegadores**

In [ ]:
# Reutiliza eventos_por_tipo_usuario cacheado
total_usuarios = eventos_por_tipo_usuario.count()
compradores = eventos_por_tipo_usuario.filter(col("purchase") > 0).count()
navegadores = total_usuarios - compradores

print(f"Total usuarios: {total_usuarios:,}")
print(f"Compradores: {compradores:,} ({compradores/total_usuarios*100:.2f}%)")
print(f"Navegadores: {navegadores:,} ({navegadores/total_usuarios*100:.2f}%)")
print(f"Ratio: {compradores/navegadores:.2f}")

Total de usuarios: 3,022,290
Compradores: 347,118 (11.49%)
Solo navegadores: 2,675,172 (88.51%)
Ratio compradores/navegadores: 0.13


**Análisis de compras por usuario**

In [49]:
from pyspark.sql.functions import count, avg, min as spark_min, max as spark_max

compras_por_usuario = df_clean.filter(col("event_type") == "purchase") \
    .groupBy("user_id").agg(
        count("*").alias("num_compras"),
        spark_sum("price").alias("total_gastado"),
        avg("price").alias("gasto_promedio"),
        spark_min("price").alias("gasto_minimo"),
        spark_max("price").alias("gasto_maximo")
    )
    
print("Estadísticas de compras por usuario:")
compras_por_usuario.describe().show()

Estadísticas de compras por usuario:
+-------+--------------------+------------------+------------------+-----------------+-----------------+-----------------+
|summary|             user_id|       num_compras|     total_gastado|   gasto_promedio|     gasto_minimo|     gasto_maximo|
+-------+--------------------+------------------+------------------+-----------------+-----------------+-----------------+
|  count|              347118|            347118|            347118|           347118|           347118|           347118|
|   mean| 5.359970205283938E8|2.1398285309318443| 662.4064803035133|278.0298408672442|241.1867638670557|320.2669899285127|
| stddev|1.8498600835749403E7| 3.638736921435989| 2074.214191061826|311.2153181859074|297.9926696586471|  360.88576705247|
|    min|           264649825|                 1|              0.88|             0.88|             0.77|             0.88|
|    max|           566278294|               321|265569.51999999996|          2574.04|          2574.0

## Tasa de conversión por usuario

In [31]:
# Calculo de tasa de conversión por usuario
conversion_por_usuario = eventos_por_tipo_usuario.withColumn(
    "tasa_de_conversion",
    when(col("view") > 0, col("purchase") / col("view") * 100).otherwise(0)
).withColumn(
    "cart_rate",
    when(col("view") > 0, col("cart") / col("view") * 100).otherwise(0)
)

print("Distribución de tasas de conversión por usuario:")
conversion_por_usuario.select("tasa_de_conversion", "cart_rate").describe().show()

Distribución de tasas de conversión por usuario:
+-------+------------------+------------------+
|summary|tasa_de_conversion|         cart_rate|
+-------+------------------+------------------+
|  count|           3022290|           3022290|
|   mean| 2.003747909988533| 2.527292187227874|
| stddev| 8.888059905115897|13.473155834735463|
|    min|               0.0|               0.0|
|    max|             200.0|            2800.0|
+-------+------------------+------------------+



## 1.4 Análisis de productos y categorías

**Catálogo de productos**

In [34]:
# Total de productos únicos
total_productos = df_clean.select("product_id").distinct().count()
print(f"Total de productos únicos: {total_productos:,}")

Total de productos únicos: 166,794


**Tasa de conversión por producto**

In [ ]:
# Conversión por producto en una sola pasada (evita join)
conversion_producto = df_clean.groupBy("product_id").agg(
    spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("num_views"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("num_purchases")
).withColumn(
    "tasa_conversion_producto",
    when(col("num_views") > 0, col("num_purchases") / col("num_views") * 100).otherwise(0)
).cache()

print("Distribución de tasas de conversión por producto:")
conversion_producto.describe("tasa_conversion_producto").show()

Distribución de tasas de conversión por producto:
+-------+------------------------+
|summary|tasa_conversion_producto|
+-------+------------------------+
|  count|                  166794|
|   mean|      0.5676354841306908|
| stddev|       2.236626626845516|
|    min|                     0.0|
|    max|                   100.0|
+-------+------------------------+



**Categorías más populares**

In [ ]:
# Categorías populares en una sola pasada
stats_categoria = df_clean.groupBy("category_code_clean").agg(
    spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("views"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
    spark_sum(when(col("event_type") == "purchase", col("price")).otherwise(0)).alias("ganancias")
).cache()

print("Top 10 categorías más vistas:")
stats_categoria.orderBy(col("views").desc()).show(10, truncate=False)
print("Top 10 categorías con más compras:")
stats_categoria.orderBy(col("purchases").desc()).show(10, truncate=False)
print("Top categorías por ganancias:")
stats_categoria.orderBy(col("ganancias").desc()).show(10, truncate=False)

Top 10 categorías más vistas:
+--------------------------------+--------+
|category_code_clean             |count   |
+--------------------------------+--------+
|unknown                         |13235871|
|electronics.smartphone          |10618648|
|electronics.clocks              |1272733 |
|computers.notebook              |1106321 |
|electronics.video.tv            |1055922 |
|electronics.audio.headphone     |1018478 |
|appliances.kitchen.refrigerators|863358  |
|appliances.kitchen.washer       |831234  |
|appliances.environment.vacuum   |772001  |
|apparel.shoes                   |759637  |
+--------------------------------+--------+
only showing top 10 rows

Top 10 categorías con más compras:
+--------------------------------+------+
|category_code_clean             |count |
+--------------------------------+------+
|electronics.smartphone          |337979|
|unknown                         |173411|
|electronics.audio.headphone     |30501 |
|electronics.video.tv            |21561 |

**Lealtad a la marca**

In [54]:
from pyspark.sql.functions import countDistinct

# Usuarios que compran de una sola marca vs varias
marcas_por_usuario = df_clean.filter(col("event_type") == "purchase").groupBy("user_id") \
    .agg(countDistinct("brand_clean").alias("num_marcas_compradas")).cache()

print("Lealtad a la marca:")
marcas_por_usuario.describe("num_marcas_compradas").show()

lealtad = marcas_por_usuario.agg(
    spark_sum(when(col("num_marcas_compradas") == 1, 1).otherwise(0)).alias("leales"),
    spark_sum(when(col("num_marcas_compradas") > 1, 1).otherwise(0)).alias("diversos")
).collect()[0]

total = lealtad["leales"] + lealtad["diversos"]
print(f"Una sola marca: {lealtad['leales']:,} ({lealtad['leales']/total*100:.2f}%)")
print(f"Varias marcas: {lealtad['diversos']:,} ({lealtad['diversos']/total*100:.2f}%)")

Lealtad a la marca
+-------+--------------------+
|summary|num_marcas_compradas|
+-------+--------------------+
|  count|              347118|
|   mean|  1.3341054050783883|
| stddev|  0.8559955198932354|
|    min|                   1|
|    max|                  30|
+-------+--------------------+

Compradores de una sola marca: 273,457 (78.78%)
Compradores de varias marcas: 73,661 (21.22%)


# 2. Análisis de Precios y Valor

## 2.1 Distribución de Precios

In [60]:
# Estadísticas descriptivas de precios
from pyspark.sql.functions import expr, percentile_approx

print("Estadísticas globales de precios:")
df_clean.select("price").describe().show()

# Percentiles para detectar outliers
df_clean.select(
    percentile_approx("price", 0.25).alias("Q1"),
    percentile_approx("price", 0.50).alias("mediana"),
    percentile_approx("price", 0.75).alias("Q3"),
    percentile_approx("price", 0.95).alias("P95"),
    percentile_approx("price", 0.99).alias("P99")
).show()

Estadísticas globales de precios:
+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          42418544|
|   mean| 290.3132680178888|
| stddev|358.29733367226834|
|    min|               0.0|
|    max|           2574.07|
+-------+------------------+

+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          42418544|
|   mean| 290.3132680178888|
| stddev|358.29733367226834|
|    min|               0.0|
|    max|           2574.07|
+-------+------------------+

+----+-------+------+-------+-------+
|  Q1|mediana|    Q3|    P95|    P99|
+----+-------+------+-------+-------+
|65.9| 162.68|358.57|1010.05|1741.33|
+----+-------+------+-------+-------+

+----+-------+------+-------+-------+
|  Q1|mediana|    Q3|    P95|    P99|
+----+-------+------+-------+-------+
|65.9| 162.68|358.57|1010.05|1741.33|
+----+-------+------+-------+-------+



__Segmentación de precios por cuantiles dentro de cada categoría__

In [ ]:
# Segmentación de precios por cuantiles dentro de cada categoría
from pyspark.sql.functions import percentile_approx

percentiles_por_categoria = df_clean.groupBy("category_code_clean").agg(
    percentile_approx("price", 0.33).alias("p33"),
    percentile_approx("price", 0.66).alias("p66")
).cache()

df_segmentado = df_clean.join(percentiles_por_categoria, "category_code_clean", "left").withColumn(
    "segmento_precio",
    when(col("price") < col("p33"), "barato")
    .when(col("price") < col("p66"), "medio")
    .otherwise("caro")
).cache()

print("Distribución de segmentos de precio:")
df_segmentado.groupBy("segmento_precio").count().orderBy("segmento_precio").show()

Distribución de segmentos de precio:
+---------------+--------+
|segmento_precio|   count|
+---------------+--------+
|         barato|13857551|
|           caro|14551579|
|          medio|14009414|
+---------------+--------+

+---------------+--------+
|segmento_precio|   count|
+---------------+--------+
|         barato|13857551|
|           caro|14551579|
|          medio|14009414|
+---------------+--------+



In [62]:
# Ejemplo: Ver umbrales por categoría
print("Umbrales de precio por categoría (muestra):")
percentiles_por_categoria.orderBy(col("p66").desc()).show(15, truncate=False)

Umbrales de precio por categoría (muestra):
+--------------------------------+------+------+
|category_code_clean             |p33   |p66   |
+--------------------------------+------+------+
|electronics.camera.photo        |442.71|900.41|
|computers.notebook              |420.6 |720.71|
|electronics.video.projector     |402.1 |705.72|
|computers.desktop               |173.75|617.75|
|furniture.living_room.sofa      |437.33|592.01|
|auto.accessories.winch          |344.77|573.14|
|electronics.smartphone          |215.57|485.54|
|electronics.video.tv            |269.2 |462.05|
|appliances.kitchen.dishwasher   |324.04|445.29|
|electronics.camera.video        |179.93|437.38|
|sport.trainer                   |200.78|383.53|
|kids.skates                     |257.15|368.81|
|appliances.kitchen.washer       |231.64|360.34|
|appliances.kitchen.refrigerators|242.63|360.32|
|electronics.tablet              |115.81|360.09|
+--------------------------------+------+------+
only showing top 15 rows


In [77]:
# Ejecutar si no se ejecutaron las bibliotecas antes
from pyspark.sql.functions import sum as spark_sum, when, col, count

In [68]:
# Conversión por segmento de precio
conversion_por_segmento = df_segmentado.groupBy("segmento_precio").agg(
    spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("views"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("purchases")
).withColumn(
    "tasa_conversion", col("purchases") / col("views") * 100
)
print("Tasa de conversión por segmento de precio:")
conversion_por_segmento.orderBy("segmento_precio").show()

Tasa de conversión por segmento de precio:
+---------------+--------+---------+------------------+
|segmento_precio|   views|purchases|   tasa_conversion|
+---------------+--------+---------+------------------+
|         barato|13250908|   268130|2.0234839755886918|
|           caro|14059397|   230513|1.6395653383996485|
|          medio|13467023|   244130|1.8127985672854348|
+---------------+--------+---------+------------------+

+---------------+--------+---------+------------------+
|segmento_precio|   views|purchases|   tasa_conversion|
+---------------+--------+---------+------------------+
|         barato|13250908|   268130|2.0234839755886918|
|           caro|14059397|   230513|1.6395653383996485|
|          medio|13467023|   244130|1.8127985672854348|
+---------------+--------+---------+------------------+



__Precios por categoría__

In [78]:
# Ejecutar si no se ejecutaron las bibliotecas antes
from pyspark.sql.functions import avg, percentile_approx, min, max, split, col

In [72]:
# Estadísticas de precio por categoría principal
from pyspark.sql.functions import split

df_con_cat_principal = df_clean.withColumn(
    "categoria_principal", split(col("category_code_clean"), "\\.").getItem(0)
)

print("Precios por categoría principal:")
df_con_cat_principal.groupBy("categoria_principal").agg(
    avg("price").alias("precio_promedio"),
    percentile_approx("price", 0.5).alias("mediana"),
    min("price").alias("min"),
    max("price").alias("max")
).orderBy(col("precio_promedio").desc()).show(15)

Precios por categoría principal:
+-------------------+------------------+-------+---+-------+
|categoria_principal|   precio_promedio|mediana|min|    max|
+-------------------+------------------+-------+---+-------+
|          computers| 504.3028990900214| 360.34|0.0|2574.04|
|        electronics|412.26495465454116| 250.91|0.0|2574.07|
|              sport|  387.200470205765|  205.9|0.0|2573.81|
|       country_yard| 270.4435302890421| 238.86|0.0|2426.32|
|          furniture| 266.1417113404556| 176.53|0.0|2574.04|
|         appliances|223.47014463231133| 151.84|0.0|2574.04|
|            unknown|184.92899572328912|  82.34|0.0|2574.04|
|       construction| 164.2868837074333|  90.07|0.0|2571.19|
|               kids|160.57896687871408|  79.67|0.0|2571.49|
|               auto|140.92986071086915| 121.24|0.0|2165.76|
|            apparel| 81.59048442511641|  73.36|0.0| 913.79|
|        accessories| 60.90159841157904|  43.76|0.0|1717.16|
|           medicine| 50.95741519124206|  42.68|0.0|

__Variación de precios en el tiempo (detección de promociones)__

In [79]:
# Ejecutar si faltan bibliotecas
from pyspark.sql.functions import to_date, avg, percentile_approx, expr, col

In [ ]:
# Precio promedio por día (detectar promociones)
precio_por_dia = df_clean.withColumn("fecha", to_date(col("event_time"))).groupBy("fecha").agg(
    avg("price").alias("precio_promedio"),
    percentile_approx("price", 0.5).alias("mediana_precio")
).orderBy("fecha")

print("Variación de precio promedio por día:")
precio_por_dia.show(31)

# Detectar días con precios anómalos
stats_precio = precio_por_dia.agg(
    avg("precio_promedio").alias("media"),
    expr("stddev(precio_promedio)").alias("std")
).collect()[0]

dias_anomalos = precio_por_dia.filter(
    (col("precio_promedio") < stats_precio["media"] - 2 * stats_precio["std"]) |
    (col("precio_promedio") > stats_precio["media"] + 2 * stats_precio["std"])
)
print("Días con precios anómalos (±2):")
dias_anomalos.show()

Variación de precio promedio por día:
+----------+------------------+--------------+
|     fecha|   precio_promedio|mediana_precio|
+----------+------------------+--------------+
|2019-10-01|297.88166849318367|         161.9|
|2019-10-02|300.13027390450395|        164.17|
|2019-10-03|301.16020407873447|        168.84|
|2019-10-04|298.95175895995686|        169.86|
|2019-10-05|297.39146986385776|        167.29|
|2019-10-06| 301.1398038892656|        168.83|
|2019-10-07|296.06396959434045|        160.81|
|2019-10-08|277.88350016823733|        148.65|
|2019-10-09|281.78438398614804|        151.36|
|2019-10-10|289.72040033349964|         159.3|
|2019-10-11|282.06842161501646|        153.67|
|2019-10-12|281.92445732427416|        153.93|
|2019-10-13|279.54829235971624|        153.84|
|2019-10-14|298.04799310560736|        172.19|
|2019-10-15| 296.0082195538297|        170.92|
|2019-10-16|291.07615194758506|        167.31|
|2019-10-17|292.10875402958476|        166.23|
|2019-10-18| 287.70531

## 2.2 Análisis de Revenue

In [ ]:
# Revenue con cache para reutilizar
compras = df_clean.filter(col("event_type") == "purchase").cache()
revenue_total = compras.agg(spark_sum("price")).collect()[0][0]
print(f"Revenue total: ${revenue_total:,.2f}")

revenue_por_dia = compras.withColumn("fecha", to_date(col("event_time"))).groupBy("fecha").agg(
    spark_sum("price").alias("revenue"),
    count("*").alias("transacciones")
).orderBy("fecha")

print("\nRevenue por día:")
revenue_por_dia.show(31)
revenue_por_dia.describe("revenue").show()

Revenue total: $229,933,212.63

Revenue por día:

Revenue por día:
+----------+------------------+-------------+
|     fecha|           revenue|transacciones|
+----------+------------------+-------------+
|2019-10-01|6275579.0600000005|        19305|
|2019-10-02|        6213628.53|        19469|
|2019-10-03|6233782.9799999995|        19255|
|2019-10-04| 8623058.189999998|        27039|
|2019-10-05| 7341094.459999996|        23492|
|2019-10-06| 6737258.170000001|        22169|
|2019-10-07| 6348189.059999999|        21378|
|2019-10-08|        6819701.26|        23071|
|2019-10-09|        6855326.05|        22747|
|2019-10-10| 6665413.209999999|        21992|
|2019-10-11|        7716208.55|        26224|
|2019-10-12| 7307691.569999997|        25373|
|2019-10-13|        8457606.49|        29561|
|2019-10-14| 9356691.649999999|        28405|
|2019-10-15| 8652872.260000002|        26371|
|2019-10-16| 9747164.719999991|        31393|
|2019-10-17| 9026019.069999998|        28317|
|2019-10-18| 

__Revenue por categoría y marca__

In [81]:
# Revenue por categoría y marca (reutiliza compras cacheado)
revenue_categoria = compras.groupBy("category_code_clean").agg(
    spark_sum("price").alias("revenue"), count("*").alias("transacciones")
).withColumn("contribucion_pct", col("revenue") / revenue_total * 100).orderBy(col("revenue").desc())

print("Top 15 categorías por revenue:")
revenue_categoria.show(15, truncate=False)

revenue_marca = compras.groupBy("brand_clean").agg(
    spark_sum("price").alias("revenue"), count("*").alias("transacciones")
).withColumn("contribucion_pct", col("revenue") / revenue_total * 100).orderBy(col("revenue").desc())

print("Top 15 marcas por revenue:")
revenue_marca.show(15)

Top 15 categorías por revenue:
+--------------------------------+--------------------+-------------+-------------------+
|category_code_clean             |revenue             |transacciones|contribucion_pct   |
+--------------------------------+--------------------+-------------+-------------------+
|electronics.smartphone          |1.5703391396999988E8|337979       |68.29544639238053  |
|unknown                         |2.292216508000001E7 |173411       |9.96905354290227   |
|computers.notebook              |8978883.419999996   |15588        |3.9049962888347443 |
|electronics.video.tv            |8422119.379999999   |21561        |3.6628546540392866 |
|electronics.clocks              |4817089.040000001   |17903        |2.094994883471439  |
|appliances.kitchen.washer       |4658223.46          |16146        |2.025902829225391  |
|appliances.kitchen.refrigerators|3830077.0099999984  |11218        |1.6657345697001238 |
|electronics.audio.headphone     |3538807.1700000004  |30501        |

__Distribución de revenue por usuario (identificar ballenas)__

In [82]:
# LTV por usuario (reutiliza compras cacheado)
ltv_usuario = compras.groupBy("user_id").agg(
    spark_sum("price").alias("ltv"), count("*").alias("num_compras")
).orderBy(col("ltv").desc()).cache()

print("Distribución de LTV:")
ltv_usuario.describe("ltv").show()
print("Percentiles de LTV:")
ltv_usuario.selectExpr(
    "percentile_approx(ltv, 0.50) as p50",
    "percentile_approx(ltv, 0.75) as p75",
    "percentile_approx(ltv, 0.90) as p90",
    "percentile_approx(ltv, 0.95) as p95",
    "percentile_approx(ltv, 0.99) as p99"
).show()
print("Top 20 ballenas:")
ltv_usuario.show(20)

Distribución de LTV:
+-------+------------------+
|summary|               ltv|
+-------+------------------+
|  count|            347118|
|   mean|  662.406480303528|
| stddev|2074.2141910618398|
|    min|              0.88|
|    max|265569.51999999984|
+-------+------------------+

Percentiles de LTV:
+-------+------------------+
|summary|               ltv|
+-------+------------------+
|  count|            347118|
|   mean|  662.406480303528|
| stddev|2074.2141910618398|
|    min|              0.88|
|    max|265569.51999999984|
+-------+------------------+

Percentiles de LTV:
+------+------+-------+-------+-------+
|   p50|   p75|    p90|    p95|    p99|
+------+------+-------+-------+-------+
|246.52|594.53|1418.05|2336.73|6667.38|
+------+------+-------+-------+-------+

Top 20 ballenas:
+---------+------------------+-----------+
|  user_id|               ltv|num_compras|
+---------+------------------+-----------+
|519267944|265569.51999999984|        183|
|513117637|244499.9999999

__Análisis de Pareto (regla 80/20)__

In [83]:
# Análisis de Pareto (reutiliza ltv_usuario cacheado)
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

total_compradores = ltv_usuario.count()
w = Window.orderBy(col("ltv").desc())

pareto = ltv_usuario.withColumn("rank", row_number().over(w)) \
    .withColumn("ltv_acum", spark_sum("ltv").over(Window.orderBy(col("ltv").desc()).rowsBetween(Window.unboundedPreceding, 0))) \
    .withColumn("pct_revenue", col("ltv_acum") / revenue_total * 100) \
    .withColumn("pct_usuarios", col("rank") / total_compradores * 100)

usuarios_80pct = pareto.filter(col("pct_revenue") >= 80).select("rank", "pct_usuarios").first()
print(f"Total compradores: {total_compradores:,}")
print(f"80% revenue generado por {usuarios_80pct['rank']:,} usuarios ({usuarios_80pct['pct_usuarios']:.2f}%)")

pareto.withColumn("segmento",
    when(col("pct_revenue") <= 50, "top_50pct")
    .when(col("pct_revenue") <= 80, "mid_30pct")
    .otherwise("bottom_20pct")
).groupBy("segmento").agg(
    count("*").alias("usuarios"), spark_sum("ltv").alias("revenue")
).withColumn("pct_usuarios", col("usuarios") / total_compradores * 100) \
 .withColumn("pct_revenue", col("revenue") / revenue_total * 100).show()

Total compradores: 347,118
80% revenue generado por 98,087 usuarios (28.26%)
+------------+--------+--------------------+-----------------+------------------+
|    segmento|usuarios|             revenue|     pct_usuarios|       pct_revenue|
+------------+--------+--------------------+-----------------+------------------+
|   top_50pct|   25406|1.1496487390000086E8|7.319124908532546| 49.99924655730277|
|   mid_30pct|   72680| 6.898132837000144E7|20.93812478753623|30.000593468418884|
|bottom_20pct|  249032|4.5987010360003255E7|71.74275030393122|20.000159974280827|
+------------+--------+--------------------+-----------------+------------------+

+------------+--------+--------------------+-----------------+------------------+
|    segmento|usuarios|             revenue|     pct_usuarios|       pct_revenue|
+------------+--------+--------------------+-----------------+------------------+
|   top_50pct|   25406|1.1496487390000086E8|7.319124908532546| 49.99924655730277|
|   mid_30pct|   726

## 1.5 Análisis de precios y valor

## 1.6 Análisis de sesiones y engagement
Las sesiones ya están definidas por el creador del dataset: una nueva sesión inicia cuando el usuario está inactivo por más de 2 horas.

In [44]:
from pyspark.sql.functions import unix_timestamp, lag, when
from pyspark.sql.window import Window

print(f"Total de sesiones únicas: {df_clean.select('user_session').distinct().count():,}")

Total de sesiones únicas: 9,244,422


**Estadísticas de sesiones**

In [55]:
# Métricas por sesión
sesiones = df_clean.groupBy("user_session", "user_id").agg(
    count("*").alias("num_eventos"),
    spark_min("event_time").alias("inicio_sesion"),
    spark_max("event_time").alias("fin_sesion"),
    countDistinct("product_id").alias("productos_vistos"),
    countDistinct("category_code_clean").alias("categorias_vistas"),
    countDistinct("brand_clean").alias("marcas_vistas")
)

# Duración de sesiones (en minutos)
sesiones = sesiones.withColumn(
    "duracion_minutos",
    (unix_timestamp("fin_sesion") - unix_timestamp("inicio_sesion")) / 60
)

print("Estadísticas de sesiones:")
sesiones.describe("num_eventos", "duracion_minutos", "productos_vistos").show()

Estadísticas de sesiones:
+-------+-----------------+------------------+------------------+
|summary|      num_eventos|  duracion_minutos|  productos_vistos|
+-------+-----------------+------------------+------------------+
|  count|          9244772|           9244772|           9244772|
|   mean|4.588381844354842|17.363946889838395|3.0056153899739226|
| stddev|6.759781302523697|375.35593323265954| 4.169674792746022|
|    min|                1|               0.0|                 1|
|    max|             1159|44008.816666666666|               868|
+-------+-----------------+------------------+------------------+



In [56]:
# Percentiles de duración de sesiones
print("Percentiles de duración de sesiones (minutos):")
sesiones.selectExpr(
    "percentile_approx(duracion_minutos, 0.25) as p25",
    "percentile_approx(duracion_minutos, 0.50) as p50_median",
    "percentile_approx(duracion_minutos, 0.75) as p75",
    "percentile_approx(duracion_minutos, 0.90) as p90",
    "percentile_approx(duracion_minutos, 0.95) as p95",
    "percentile_approx(duracion_minutos, 0.99) as p99"
).show()

print("\nPercentiles de eventos por sesión:")
sesiones.selectExpr(
    "percentile_approx(num_eventos, 0.25) as p25",
    "percentile_approx(num_eventos, 0.50) as p50_median",
    "percentile_approx(num_eventos, 0.75) as p75",
    "percentile_approx(num_eventos, 0.90) as p90",
    "percentile_approx(num_eventos, 0.95) as p95",
    "percentile_approx(num_eventos, 0.99) as p99"
).show()

Percentiles de duración de sesiones (minutos):
+---+----------+-----------------+------------------+------------------+-----+
|p25|p50_median|              p75|               p90|               p95|  p99|
+---+----------+-----------------+------------------+------------------+-----+
|0.0|      1.05|4.483333333333333|11.933333333333334|21.083333333333332|78.35|
+---+----------+-----------------+------------------+------------------+-----+


Percentiles de eventos por sesión:
+---+----------+---+---+---+---+
|p25|p50_median|p75|p90|p95|p99|
+---+----------+---+---+---+---+
|  1|         2|  5| 10| 16| 32|
+---+----------+---+---+---+---+



**Sesiones por usuario**

In [57]:
# Número de sesiones por usuario
sesiones_por_usuario = sesiones.groupBy("user_id").agg(
    count("*").alias("num_sesiones"),
    avg("duracion_minutos").alias("duracion_promedio_sesion"),
    avg("num_eventos").alias("eventos_promedio_sesion"),
    avg("productos_vistos").alias("productos_promedio_sesion"),
    spark_sum("num_eventos").alias("total_eventos")
)

print("Estadísticas de sesiones por usuario:")
sesiones_por_usuario.describe().show()

Estadísticas de sesiones por usuario:
+-------+--------------------+------------------+------------------------+-----------------------+-------------------------+-----------------+
|summary|             user_id|      num_sesiones|duracion_promedio_sesion|eventos_promedio_sesion|productos_promedio_sesion|    total_eventos|
+-------+--------------------+------------------+------------------------+-----------------------+-------------------------+-----------------+
|  count|             3022290|           3022290|                 3022290|                3022290|                  3022290|          3022290|
|   mean| 5.404673750553795E8|3.0588633122566002|      23.446607718515963|      3.807435730685657|       2.6015834281689365| 14.0352328863213|
| stddev|1.9471434388504434E7| 6.757154813024777|       460.8154730386438|      4.263126404916929|        2.763128370767271|32.75705325743531|
|    min|            33869381|                 1|                     0.0|                    1.0|      

In [58]:
# Percentiles de sesiones por usuario
print("Percentiles de número de sesiones por usuario:")
sesiones_por_usuario.selectExpr(
    "percentile_approx(num_sesiones, 0.25) as p25",
    "percentile_approx(num_sesiones, 0.50) as p50_median",
    "percentile_approx(num_sesiones, 0.75) as p75",
    "percentile_approx(num_sesiones, 0.90) as p90",
    "percentile_approx(num_sesiones, 0.95) as p95"
).show()

Percentiles de número de sesiones por usuario:
+---+----------+---+---+---+
|p25|p50_median|p75|p90|p95|
+---+----------+---+---+---+
|  1|         2|  3|  7| 10|
+---+----------+---+---+---+



**Tipos de sesiones según comportamiento**

In [59]:
# Clasificación de sesiones según eventos que contienen
sesiones_con_eventos = df_clean.groupBy("user_session").agg(
    spark_sum(when(col("event_type") == "view", 1).otherwise(0)).alias("num_views"),
    spark_sum(when(col("event_type") == "cart", 1).otherwise(0)).alias("num_carts"),
    spark_sum(when(col("event_type") == "purchase", 1).otherwise(0)).alias("num_purchases")
)

# Clasificación de sesiones
sesiones_clasificadas = sesiones_con_eventos.withColumn(
    "tipo_sesion",
    when(col("num_purchases") > 0, "compra") \
    .when(col("num_carts") > 0, "carrito_sin_compra") \
    .otherwise("solo_navegacion")
)

print("Distribución de tipos de sesiones:")
sesiones_clasificadas.groupBy("tipo_sesion").count() \
    .withColumn("porcentaje", col("count") / sesiones_clasificadas.count() * 100) \
    .orderBy(col("count").desc()) \
    .show()

Distribución de tipos de sesiones:
+------------------+-------+-----------------+
|       tipo_sesion|  count|       porcentaje|
+------------------+-------+-----------------+
|   solo_navegacion|8333625|90.14760468529022|
|            compra| 629560| 6.81016076505378|
|carrito_sin_compra| 281237|3.042234549655998|
+------------------+-------+-----------------+



**Tasa de conversión por sesión**

In [60]:
# Tasa de conversión a nivel de sesión
sesiones_conversion = sesiones_con_eventos.withColumn(
    "conversion_sesion",
    when(col("num_views") > 0, col("num_purchases") / col("num_views") * 100).otherwise(0)
).withColumn(
    "cart_rate_sesion",
    when(col("num_views") > 0, col("num_carts") / col("num_views") * 100).otherwise(0)
)

print("Distribución de tasas de conversión por sesión:")
sesiones_conversion.select("conversion_sesion", "cart_rate_sesion").describe().show()

Distribución de tasas de conversión por sesión:
+-------+------------------+------------------+
|summary| conversion_sesion|  cart_rate_sesion|
+-------+------------------+------------------+
|  count|           9244422|           9244422|
|   mean| 2.943402968645941|3.2997233558860133|
| stddev|13.542354771047405|19.894352061672844|
|    min|               0.0|               0.0|
|    max|             500.0|            4300.0|
+-------+------------------+------------------+



**Análisis de engagement por usuario**

In [62]:
# Métricas de engagement por usuario
engagement_usuario = df_clean.groupBy("user_id").agg(
    countDistinct("user_session").alias("num_sesiones"),
    countDistinct(to_date("event_time")).alias("dias_activos"),
    countDistinct("product_id").alias("productos_unicos_vistos"),
    countDistinct("category_code_clean").alias("categorias_unicas"),
    countDistinct("brand_clean").alias("marcas_unicas"),
    count("*").alias("total_eventos")
)

# Días entre primera y última actividad
actividad_usuario = df_clean.groupBy("user_id").agg(
    spark_min("event_time").alias("primera_actividad"),
    spark_max("event_time").alias("ultima_actividad")
).withColumn(
    "dias_span",
    (unix_timestamp("ultima_actividad") - unix_timestamp("primera_actividad")) / 86400
)

engagement_completo = engagement_usuario.join(actividad_usuario, "user_id")

print("Estadísticas de engagement por usuario:")
engagement_completo.describe("num_sesiones", "dias_activos", "productos_unicos_vistos").show()

Estadísticas de engagement por usuario:
+-------+-----------------+------------------+-----------------------+
|summary|     num_sesiones|      dias_activos|productos_unicos_vistos|
+-------+-----------------+------------------+-----------------------+
|  count|          3022290|           3022290|                3022290|
|   mean|3.058862650506735|2.1419926611939952|      7.711910504948235|
| stddev|6.757154720855969| 2.179535480859494|     15.158596341254642|
|    min|                1|                 1|                      1|
|    max|             7400|                31|                   1174|
+-------+-----------------+------------------+-----------------------+



**Diversidad de exploración por usuario**

In [63]:
# Usuarios mono-categoría vs multi-categoría
usuarios_mono = engagement_completo.filter(col("categorias_unicas") == 1).count()
usuarios_multi = engagement_completo.filter(col("categorias_unicas") > 1).count()

print(f"Usuarios que exploran 1 sola categoría: {usuarios_mono:,} ({usuarios_mono/(usuarios_mono+usuarios_multi)*100:.2f}%)")
print(f"Usuarios que exploran múltiples categorías: {usuarios_multi:,} ({usuarios_multi/(usuarios_mono+usuarios_multi)*100:.2f}%)")

print("\nPercentiles de categorías exploradas por usuario:")
engagement_completo.selectExpr(
    "percentile_approx(categorias_unicas, 0.25) as p25",
    "percentile_approx(categorias_unicas, 0.50) as p50_median",
    "percentile_approx(categorias_unicas, 0.75) as p75",
    "percentile_approx(categorias_unicas, 0.90) as p90",
    "percentile_approx(categorias_unicas, 0.95) as p95"
).show()

Usuarios que exploran 1 sola categoría: 1,808,915 (59.85%)
Usuarios que exploran múltiples categorías: 1,213,375 (40.15%)

Percentiles de categorías exploradas por usuario:
+---+----------+---+---+---+
|p25|p50_median|p75|p90|p95|
+---+----------+---+---+---+
|  1|         1|  2|  4|  5|
+---+----------+---+---+---+



**Análisis de rutas de navegación (secuencias de categorías)**

In [64]:
from pyspark.sql.functions import collect_list, concat_ws, row_number

# Top secuencias de categorías por sesión (primeras 5 categorías visitadas)
windowSpec = Window.partitionBy("user_session").orderBy("event_time")

secuencias = df_clean.filter(col("event_type") == "view") \
    .withColumn("row_num", row_number().over(windowSpec)) \
    .filter(col("row_num") <= 5) \
    .groupBy("user_session").agg(
        collect_list("category_code_clean").alias("secuencia_categorias")
    )

# Convertir array a string para análisis
secuencias_str = secuencias.withColumn(
    "secuencia_str",
    concat_ws(" -> ", col("secuencia_categorias"))
)

print("Top 15 rutas de navegación más comunes:")
secuencias_str.groupBy("secuencia_str").count() \
    .orderBy(col("count").desc()) \
    .show(15, truncate=100)

Top 15 rutas de navegación más comunes:
+----------------------------------------------------------------------------------------------------+-------+
|                                                                                       secuencia_str|  count|
+----------------------------------------------------------------------------------------------------+-------+
|                                                                              electronics.smartphone|1149090|
|                                                                                             unknown|1083459|
|                                                 unknown -> unknown -> unknown -> unknown -> unknown| 632980|
|electronics.smartphone -> electronics.smartphone -> electronics.smartphone -> electronics.smartph...| 578084|
|                                                    electronics.smartphone -> electronics.smartphone| 546676|
|                                                                       

**Profundidad de sesión (bounce vs engaged)**

In [65]:
# Sesiones bounce (solo un evento) vs engaged (múltiples eventos)
sesiones_bounce = sesiones.filter(col("num_eventos") == 1).count()
sesiones_engaged = sesiones.filter(col("num_eventos") > 1).count()
total_sesiones = sesiones.count()

print(f"Total de sesiones: {total_sesiones:,}")
print(f"Sesiones bounce (1 evento): {sesiones_bounce:,} ({sesiones_bounce / total_sesiones * 100:.2f}%)")
print(f"Sesiones engaged (>1 evento): {sesiones_engaged:,} ({sesiones_engaged / total_sesiones * 100:.2f}%)")   

# Bounce rate
bounce_rate = sesiones_bounce / total_sesiones * 100
print(f"Bounce rate: {bounce_rate:.2f}%")


Total de sesiones: 9,244,772
Sesiones bounce (1 evento): 3,271,061 (35.38%)
Sesiones engaged (>1 evento): 5,973,711 (64.62%)
Bounce rate: 35.38%


## 1.7 Análisis RFM y segmentación de usuarios.


* Recency: Días desde la última interacción
* Frequency: Número total de eventos o compras
* Monetary: Valor total gastado


**Cálculo de métricas RFM (Recency, Frequency, Monetary)**

In [67]:
from pyspark.sql.functions import lit
# Fecha de referencia para análisis temporal (último día del dataset)
fecha_referencia = df_clean.agg(spark_max("event_time")).collect()[0][0]

# Calculo RFM solo para compradores
rfm = df_clean.filter(col("event_type") == "purchase").groupBy("user_id").agg(
    # Recency: días desde la última compra
    ((unix_timestamp(lit(fecha_referencia)) - unix_timestamp(spark_max("event_time"))) / 86400).alias("recency_days"),
    # Frequency: número de compras
    count("*").alias("frequency"),
    # Monetary: gasto total
    spark_sum("price").alias("monetary")
)

print("Métricas RFM para compradores:")
rfm.describe().show()

Métricas RFM para compradores:
+-------+--------------------+--------------------+------------------+------------------+
|summary|             user_id|        recency_days|         frequency|          monetary|
+-------+--------------------+--------------------+------------------+------------------+
|  count|              347118|              347118|            347118|            347118|
|   mean| 5.359970205283938E8|  14.153950384847628|2.1398285309318443| 662.4064803035133|
| stddev|1.8498600835749403E7|    8.67657382146653| 3.638736921435989| 2074.214191061826|
|    min|           264649825|4.976851851851852E-4|                 1|              0.88|
|    max|           566278294|  30.995798611111113|               321|265569.51999999996|
+-------+--------------------+--------------------+------------------+------------------+



In [68]:
# Percentiles RFM
print("Percentiles de Recency (días desde última compra):")
rfm.selectExpr(
    "percentile_approx(recency_days, 0.25) as p25",
    "percentile_approx(recency_days, 0.5) as p50_median",
    "percentile_approx(recency_days, 0.75) as p75",
).show()

print("Percentiles de Frequency (número de compras):")
rfm.selectExpr(
    "percentile_approx(frequency, 0.25) as p25",
    "percentile_approx(frequency, 0.5) as p50_median",
    "percentile_approx(frequency, 0.75) as p75",
    "percentile_approx(frequency, 0.90) as p90"
).show()

print("Percentiles de Monetary (gasto total):")
rfm.selectExpr(
    "percentile_approx(monetary, 0.25) as p25",
    "percentile_approx(monetary, 0.5) as p50_median",
    "percentile_approx(monetary, 0.75) as p75",
    "percentile_approx(monetary, 0.90) as p90"
).show()

Percentiles de Recency (días desde última compra):
+-----------------+------------------+-----------------+
|              p25|        p50_median|              p75|
+-----------------+------------------+-----------------+
|6.636631944444445|13.678425925925925|20.88596064814815|
+-----------------+------------------+-----------------+

Percentiles de Frequency (número de compras):
+---+----------+---+---+
|p25|p50_median|p75|p90|
+---+----------+---+---+
|  1|         1|  2|  4|
+---+----------+---+---+

Percentiles de Monetary (gasto total):
+------+----------+-----+-------+
|   p25|p50_median|  p75|    p90|
+------+----------+-----+-------+
|107.59|    246.52|594.7|1418.05|
+------+----------+-----+-------+



**Segmentación RFM (scoring)**


In [69]:
from pyspark.sql.functions import concat

# Calculo de quartiles para scoring
recency_quartiles = rfm.approxQuantile("recency_days", [0.25, 0.5, 0.75], 0.001)
frequency_quartiles = rfm.approxQuantile("frequency", [0.25, 0.5, 0.75], 0.001)
monetary_quartiles = rfm.approxQuantile("monetary", [0.25, 0.5, 0.75], 0.001)

print("Recency quartiles:", recency_quartiles)
print("Frequency quartiles:", frequency_quartiles)
print("Monetary quartiles:", monetary_quartiles)

# Asignación de scores (1-4, donde 4 es mejor)
# Para Recency: menor es mejor (compró recientemente)
# Para Frequency y Monetary: mayor es mejor

rfm_scored = rfm.withColumn(
    "R_score",
    when(col("recency_days") <= recency_quartiles[0], 4)
    .when(col("recency_days") <= recency_quartiles[1], 3)
    .when(col("recency_days") <= recency_quartiles[2], 2)
    .otherwise(1)
).withColumn(
    "F_score",
    when(col("frequency") <= frequency_quartiles[0], 1)
    .when(col("frequency") <= frequency_quartiles[1], 2)
    .when(col("frequency") <= frequency_quartiles[2], 3)
    .otherwise(4)
).withColumn(
    "M_score",
    when(col("monetary") <= monetary_quartiles[0], 1)
    .when(col("monetary") <= monetary_quartiles[1], 2)
    .when(col("monetary") <= monetary_quartiles[2], 3)
    .otherwise(4)
)

# Se crea RFM_score concatenado
rfm_scored = rfm_scored.withColumn(
    "RFM_score",
    concat(col("R_score"), col("F_score"), col("M_score"))
).withColumn(
    "RFM_total",
    col("R_score") + col("F_score") + col("M_score")
)

rfm_scored.show(10)

Recency quartiles: [6.627199074074074, 13.672175925925925, 20.871180555555554]
Frequency quartiles: [1.0, 1.0, 2.0]
Monetary quartiles: [107.8, 246.52, 594.25]
+---------+------------------+---------+------------------+-------+-------+-------+---------+---------+
|  user_id|      recency_days|frequency|          monetary|R_score|F_score|M_score|RFM_score|RFM_total|
+---------+------------------+---------+------------------+-------+-------+-------+---------+---------+
|514734316|30.507395833333334|        1|             64.32|      1|      1|      1|      111|        3|
|518882191|30.354317129629628|        1|            174.27|      1|      1|      2|      112|        4|
|530054962|16.153113425925927|        2| 926.1500000000001|      2|      3|      4|      234|        9|
|513850643|        12.3684375|        4|            1893.7|      3|      4|      4|      344|       11|
|514440503| 30.61685185185185|        1|            180.16|      1|      1|      2|      112|        4|
|5371237

**Segmentos de clientes basados en RFM**

In [71]:
# Definir segmentos de clientes
rfm_segmented = rfm_scored.withColumn(
    "segmento",
    when((col("R_score") >= 4) & (col("F_score") >= 4) & (col("M_score") >= 4), "Champions")
    .when((col("R_score") >= 3) & (col("F_score") >= 3) & (col("M_score") >= 3), "Loyal Customers")
    .when((col("R_score") >= 4) & (col("F_score") <= 2), "New Customers")
    .when((col("R_score") >= 3) & (col("F_score") <= 2) & (col("M_score") <= 2), "Promising")
    .when((col("R_score") >= 3) & (col("F_score") >= 3) & (col("M_score") <= 2), "Need Attention")
    .when((col("R_score") <= 2) & (col("F_score") >= 3), "At Risk")
    .when((col("R_score") <= 2) & (col("F_score") <= 2) & (col("M_score") >= 3), "Can't Lose Them")
    .when((col("R_score") <= 1) & (col("F_score") <= 2), "Lost")
    .otherwise("Others")
)

print("Distribución de segmentos RFM:")
rfm_segmented.groupBy("segmento").count() \
    .withColumn("porcentaje", col("count") / rfm_segmented.count() * 100) \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

Distribución de segmentos RFM:
+---------------+-----+------------------+
|segmento       |count|porcentaje        |
+---------------+-----+------------------+
|Others         |54834|15.796933607591654|
|At Risk        |52542|15.136639413686412|
|Loyal Customers|43789|12.615018523960153|
|New Customers  |43733|12.598885681526168|
|Lost           |43110|12.419407809448083|
|Can't Lose Them|40314|11.613918033636976|
|Promising      |33719|9.71398775056321  |
|Champions      |19077|5.495825627020206 |
|Need Attention |16000|4.609383552567139 |
+---------------+-----+------------------+



**Estadísticas por segmento**

In [72]:
# Métricas promedio por segmento
print("Métricas promedio por segmento:")
rfm_segmented.groupBy("segmento").agg(
    count("*").alias("num_clientes"),
    avg("recency_days").alias("avg_recency"),
    avg("frequency").alias("avg_frequency"),
    avg("monetary").alias("avg_monetary"),
    spark_sum("monetary").alias("total_revenue")
).orderBy(col("total_revenue").desc()).show(truncate=False)

Métricas promedio por segmento:
+---------------+------------+------------------+------------------+------------------+--------------------+
|segmento       |num_clientes|avg_recency       |avg_frequency     |avg_monetary      |total_revenue       |
+---------------+------------+------------------+------------------+------------------+--------------------+
|Champions      |19077       |2.9360625063097143|8.924359175971064 |3631.5389233107894|6.927886803999993E7 |
|Loyal Customers|43789       |7.585379716271973 |3.5347690059147276|1191.2225497270977|5.216244422999988E7 |
|At Risk        |52542       |20.618033220028998|3.0942864755814394|938.6269171329664 |4.931733548000032E7 |
|Can't Lose Them|40314       |22.06550854266237 |1.0               |573.9881658977087 |2.3139758920000225E7|
|Others         |54834       |15.212671867000246|1.0               |252.01221067220996|1.3818837559999961E7|
|New Customers  |43733       |3.4143653446609163|1.0               |272.4614336999535 |1.1915555

**Identificación de patrones de comportamiento**

In [73]:
# Unir RFM con métricas de engagement
patrones_usuario = rfm_segmented.join(engagement_completo, "user_id", "left")

print("Vista previa de patrones combinados:")
patrones_usuario.select(
    "user_id", "segmento", "recency_days", "frequency", "monetary",
    "num_sesiones", "dias_activos", "categorias_unicas"
).show(10)

Vista previa de patrones combinados:
+---------+---------------+------------------+---------+------------------+------------+------------+-----------------+
|  user_id|       segmento|      recency_days|frequency|          monetary|num_sesiones|dias_activos|categorias_unicas|
+---------+---------------+------------------+---------+------------------+------------+------------+-----------------+
|512817507|        At Risk|29.918298611111112|        4|           1439.24|           4|           3|                1|
|513850643|Loyal Customers|        12.3684375|        4|            1893.7|           6|           5|                3|
|514440503|           Lost| 30.61685185185185|        1|            180.16|           7|           6|                3|
|514734316|           Lost|30.507395833333334|        1|             64.32|           5|           4|                4|
|517682460|        At Risk|30.689212962962962|        2|            570.87|           4|           3|                3|
|51

**Análisis de diversidad por segmento**

In [74]:
# Diversidad de categorías y productos por segmento
print("Diversidad de exploración por segmento RFM:")
patrones_usuario.groupBy("segmento").agg(
    avg("categorias_unicas").alias("avg_categorias"),
    avg("productos_unicos_vistos").alias("avg_productos"),
    avg("marcas_unicas").alias("avg_marcas"),
    avg("num_sesiones").alias("avg_sesiones"),
    avg("dias_activos").alias("avg_dias_activos")
).orderBy(col("avg_categorias").desc()).show(truncate=False)

Diversidad de exploración por segmento RFM:
+---------------+------------------+------------------+-----------------+------------------+------------------+
|segmento       |avg_categorias    |avg_productos     |avg_marcas       |avg_sesiones      |avg_dias_activos  |
+---------------+------------------+------------------+-----------------+------------------+------------------+
|Champions      |4.47171987209729  |29.73433978088798 |10.09246736908319|14.855532840593385|7.629396655658646 |
|Need Attention |4.408875          |29.39825          |12.1130625       |10.1756875        |5.476875          |
|Loyal Customers|3.5863344675603464|21.35148096553929 |7.950512685834342|9.273242138436594 |4.966407088538217 |
|At Risk        |3.4879524951467396|20.833847207947926|8.18802101176202 |8.656941113775646 |4.483556012332991 |
|Lost           |2.82577128276502  |14.421108791463698|6.532405474367896|5.552377638598933 |3.401623753189515 |
|Promising      |2.825321035617901 |14.75536047925502 |6.550

**Tipos de comportamiento de usuario**

In [78]:
# Unir con conversion_por_usuario para analizar comportamiento
comportamiento_completo = patrones_usuario.join(
    conversion_por_usuario.select("user_id", "view", "cart", "purchase", "tasa_de_conversion", "cart_rate"),
    "user_id",
    "left"
)


# Clasificar usuarios por tipo de comportamiento
usuarios_clasificados = comportamiento_completo.withColumn(
    "tipo_comportamiento",
    when((col("frequency") == 1) & (col("tasa_de_conversion") > 0), "Comprador Impulsivo")
    .when((col("view") > 50) & (col("frequency") <= 2), "Investigador")
    .when((col("frequency") >= 5) & (col("recency_days") <= 7), "Cliente Frecuente")
    .when((col("categorias_unicas") >= 5) & (col("frequency") >= 3), "Explorador")
    .when((col("view") >= 10) & (col("frequency") == 0), "Window Shopper")
    .when((col("cart") > 0) & (col("purchase") == 0), "Abandono de Carrito")
    .otherwise("Casual")
)

print("Distribución de tipos de comportamiento:")
usuarios_clasificados.groupBy("tipo_comportamiento").count() \
    .withColumn("porcentaje", col("count") / usuarios_clasificados.count() * 100) \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

Distribución de tipos de comportamiento:
+-------------------+------+-----------------+
|tipo_comportamiento|count |porcentaje       |
+-------------------+------+-----------------+
|Comprador Impulsivo|215651|62.126135780916  |
|Casual             |86894 |25.03298590104806|
|Explorador         |15653 |4.509417546770838|
|Investigador       |15010 |4.324177945252047|
|Cliente Frecuente  |13910 |4.007282826013056|
+-------------------+------+-----------------+



**Análisis temporal: Usuarios activos vs inactivos**

In [79]:
# Clasificar por recencia de actividad
usuarios_actividad = patrones_usuario.withColumn(
    "estado_actividad",
    when(col("recency_days") <= 7, "Muy Activo (última semana)")
    .when(col("recency_days") <= 14, "Activo (últimas 2 semanas)")
    .when(col("recency_days") <= 21, "En riesgo (últimas 3 semanas)")
    .otherwise("Inactivo (>3 semanas)")
)

print("Distribución de actividad de compradores:")
usuarios_actividad.groupBy("estado_actividad").count() \
    .withColumn("porcentaje", col("count") / usuarios_actividad.count() * 100) \
    .orderBy(col("count").desc()) \
    .show(truncate=False)

Distribución de actividad de compradores:
+-----------------------------+-----+------------------+
|estado_actividad             |count|porcentaje        |
+-----------------------------+-----+------------------+
|Muy Activo (última semana)   |91703|26.41839374506652 |
|Inactivo (>3 semanas)        |86510|24.92236069578645 |
|Activo (últimas 2 semanas)   |85439|24.613820084236483|
|En riesgo (últimas 3 semanas)|83466|24.045425474910548|
+-----------------------------+-----+------------------+



**Dataframes importantes para fases posteriores**

In [ ]:
'''
rfm_segmented.cache()
patrones_usuario.cache()
usuarios_actividad.cache()
'''

<!-- ## 1.6 Análisis de sesiones y engagement -->